# National Brand media and SRS sales

One-off monthly correlation input for GMA. Media is the National `Brand` total from `ius_unity_prod.media_datamart.gma_national_t1_t2_media_spend_impressions_combined`. Sales is the current National Genesis Brand total from `ius_unity_prod.srs_datamart.srs_sales_dma_model_monthly`.

Sales logic: sum every Genesis DMA-model row by month, including unassigned DMAs and VIN-undecoded Genesis units. The source is extracted at National Brand/month grain. The inner join retains only months available in both sources.

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path('/Users/maximodevries/Desktop/codex_source/dashboards/projects/gma_marcom')
MEDIA_PATH = PROJECT_ROOT / 'source' / 'media_brand_national_monthly.csv'
SRS_PATH = PROJECT_ROOT / 'source' / 'srs_brand_national_monthly.csv'
OUTPUT_PATH = PROJECT_ROOT / 'data' / 'media_brand_srs_sales_monthly.csv'

In [ ]:
media = pd.read_csv(MEDIA_PATH, parse_dates=['date_month'])
srs = pd.read_csv(SRS_PATH, parse_dates=['date_month'])

assert media[['date_month', 'model']].duplicated().sum() == 0
assert media['model'].eq('Brand').all()
assert media['geo_type'].eq('National').all()
assert srs['brand'].str.lower().eq('genesis').all()
assert srs['model'].eq('Brand').all()
assert srs[['date_month', 'model']].duplicated().sum() == 0

In [ ]:
srs_brand = srs[['date_month', 'srs_sales_volume']].copy()

joined = (
    media.drop(columns=['srs_sales_volume'], errors='ignore')
         .merge(srs_brand, on='date_month', how='inner', validate='one_to_one')
         .sort_values('date_month')
         .reset_index(drop=True)
)

joined.to_csv(OUTPUT_PATH, index=False)
{
    'rows': len(joined),
    'start': joined['date_month'].min(),
    'end': joined['date_month'].max(),
    'missing_values': int(joined.isna().sum().sum()),
    'output': OUTPUT_PATH,
}

In [ ]:
joined

In [ ]:
# Correlation-ready starting point.
joined.select_dtypes('number').corr()['srs_sales_volume'].sort_values(ascending=False)